# Hledání faktorů poptávky pomocí PROC GLMSELECT: krokový výběr, LASSO a ověřený dopředný výběr

## Shrnutí

Tým retailové analytiky používá **PROC GLMSELECT** k odhalení, které marketingové a cenové páky skutečně ovlivňují týdenní prodej kusů, a odděluje skutečné faktory poptávky od šumu. Krokový výběr hodnocený pomocí SBC, LASSO s 5násobnou křížovou validací a dopředné hledání ověřené na zadržovaném vzorku shodně nacházejí **stejnou sadu skutečných faktorů** — vlastní cenu, cenu konkurence, výdaje na reklamu, objem e-mailů, akce, svátky, region Northeast a kanál Online — a všechny čtyři podvržené rušivé proměnné (`temp_f`, `noise1`, `noise2`, `noise3`) jsou automaticky vyřazeny.

Metody se shodují i v řádech velikosti: každá odhaduje efekt vlastní ceny kolem **-28 kusů na dolar** a efekt ceny konkurence kolem **+14** — substituční znaménka, která byla zabudována do generující rovnice dat. Jediný rozdíl je na hranici významnosti — křížově validované LASSO navíc ponechává malý, staticky zanedbatelný kontrast `Region=Midwest` (odhad -8.3 se směrodatnou chybou 8.3), který krokový i dopředný výběr oba vyřazují. Seznam faktorů, který obstojí ve třech různých filozofiích výběru, je mnohem důvěryhodnější než seznam vyladěný na jedno jediné pravidlo.

## Zdroje dat

Všechna data v tomto notebooku jsou **syntetická** a generovaná přímo v kódu (žádné externí soubory, seed `20260531`). Simulují jednu sezónu týdenních prodejních panelů prodejen pro maloobchodníka se spotřebním zbožím.

| Dataset | Řádky | Granularita | Klíčové proměnné |
|---------|------|-------|---------------|
| `demand` | 100 | prodejna-týden | `units` (odezva: prodané kusy za týden); `price`, `competitor` (vlastní a konkurenční cena); `adspend`, `email` (mediální tlak); `promo`, `holiday` (příznaky událostí); `region`, `channel` (efekty CLASS); `temp_f`, `noise1`-`noise3` (podvržené/irelevantní prediktory) |

`units` je sestrojena ze známé lineární rovnice, takže „správnou“ sadu faktorů lze ověřit; `temp_f` a tři sloupce `noise` nenesou žádný skutečný signál a slouží k ověření, zda je každá metoda výběru vyřadí.

# Hledání faktorů poptávky pomocí PROC GLMSELECT

Kategorijní manažer má desítky kandidátních vysvětlujících proměnných pro týdenní tržby: vlastní cenu, cenu konkurence, výdaje na reklamu, objem e-mailů, akce, svátky, region prodejny, prodejní kanál, dokonce i počasí. Vhodit je všechny do jedné regrese znamená riskovat přeučení a nestabilní koeficienty. **PROC GLMSELECT** automatizuje hledání úsporného modelu a podporuje krokový výběr, LASSO, elastic net a least-angle výběr se zabudovanou křížovou validací a rozdělením na trénovací a ověřovací vzorek.

V tomto notebooku:

1. Vygenerujeme realistický syntetický panel týdenní poptávky prodejen se *známou* sadou skutečných faktorů (plus záměrné rušivé proměnné).
2. Spustíme **krokový výběr** hodnocený pomocí SBC.
3. Spustíme **LASSO** s 5násobnou křížovou validací.
4. Spustíme **dopředný výběr ověřený na 30% zadržovaném vzorku**.

Dobrá metoda výběru by měla najít skutečné faktory a vyřadit šum — uvidíme.

## 1. Generování syntetického panelu poptávky

Odezva `units` je sestrojena z explicitní lineární rovnice, takže známe skutečnou pravdu: záleží na ceně a ceně konkurence, výdajích na reklamu, objemu e-mailů, příznacích akce a svátku i na efektech regionu a kanálu. Proměnné `temp_f`, `noise1`, `noise2` a `noise3` jsou čisté rušivé proměnné bez skutečného vztahu k tržbám. Seed `call streaminit` zajišťuje reprodukovatelnost dat.

In [1]:
/* ---------------------------------------------------------------
   Syntetický týdenní panel poptávky prodejen pro maloobchodníka
   se spotřebním zbožím.
   'units' se řídí ZNÁMOU rovnicí; temp_f a noise1-3 jsou rušivé
   proměnné.
   --------------------------------------------------------------- */
data demand;
    CALL streaminit(20260531);
    DÉLKA region $9 channel $8 promo $3;
    OPAKUJ store_week = 1 TO 100;
        /* Mix regionů */
        u1 = rand('uniform');
        KDYŽ u1 < 0.34 PAK region = 'Northeast';
        JINAK KDYŽ u1 < 0.67 PAK region = 'Midwest';
        JINAK region = 'South';

        /* Prodejní kanál */
        KDYŽ rand('uniform') < 0.45 PAK channel = 'Store';
        JINAK channel = 'Online';

        /* Cenové a mediální faktory */
        price      = round(8  + rand('uniform') * 12, 0.01);
        competitor = round(8  + rand('uniform') * 12, 0.01);
        adspend    = round(rand('gamma', 2) * 500, 1);
        email      = round(rand('uniform') * 100, 1);

        /* Příznaky událostí a irelevantní rušivá proměnná počasí */
        temp_f     = round(40 + rand('uniform') * 50, 0.1);
        holiday    = (rand('uniform') < 0.12);
        KDYŽ rand('uniform') < 0.40 PAK promo = 'Yes';
        JINAK promo = 'No';

        /* Čisté šumové prediktory (bez skutečného efektu) */
        noise1 = rand('normal');
        noise2 = rand('normal');
        noise3 = rand('normal');

        /* Týdenní prodané kusy ze známé strukturální rovnice */
        units = 900
              - 28   * price
              + 14   * competitor
              + 0.06 * adspend
              + 1.2  * email
              + 55   * (promo = 'Yes')
              + 70   * holiday
              + 40   * (region = 'Northeast')
              + 25   * (channel = 'Online')
              + 30   * rand('normal');
        KDYŽ units < 0 PAK units = 0;
        VÝSTUP;
    KONEC;
    PONECHAT region channel promo price competitor adspend email temp_f
         holiday noise1 noise2 noise3 units;
    ŠTÍTEK region='Region'
          channel='Prodejní kanál'
          promo='Akce'
          price='Cena'
          competitor='Cena konkurence'
          adspend='Výdaje na reklamu'
          email='Objem e-mailů'
          temp_f='Teplota (F)'
          holiday='Svátek'
          noise1='Šum 1'
          noise2='Šum 2'
          noise3='Šum 3'
          units='Prodané kusy';
SPUSTIT;


NOTE: DATA demand


NOTE: Wrote demand (100 rows, 13 columns).
NOTE: DATA elapsed:
  wall  0.05 seconds
  cpu   0.05 seconds


## 2. Profilace dat

Před modelováním ověříme, že odezva a hlavní spojité kandidátní proměnné mají rozumné rozsahy hodnot.

In [2]:
PROCEDURA PRŮMĚRY data=demand n mean std MIN MAX maxdec=1;
    PROMĚNNÁ units price competitor adspend email;
    NÁZEV 'Týdenní poptávka a kandidátní faktory';
SPUSTIT;

                                         Týdenní poptávka a kandidátní faktory                                          

                                                  The MEANS Procedure

 Variable    Label                      N        Mean     Std Dev     Minimum     Maximum
 ----------------------------------------------------------------------------------------
 units       Prodané kusy             100       874.8       136.3       508.6      1162.3
 price       Cena                     100        14.0         3.4         8.0        19.9
 competitor  Cena konkurence          100        13.8         3.4         8.1        19.9
 adspend     Výdaje na reklamu        100       990.6       726.9        50.0      3358.0
 email       Objem e-mailů            100        45.5        26.4         0.0        99.0
 ----------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## 3. Krokový výběr hodnocený pomocí SBC

Krokový výběr střídavě přidává a odebírá efekty, zde posuzované pomocí **Schwarzova bayesovského kritéria (SBC)** jak pro vstupní test (`select=sbc`), tak pro volbu finálního modelu (`choose=sbc`). SBC penalizuje složitost přísněji než AIC, a upřednostňuje tak úspornější modely.

Klíčové příkazy a volby:

- `CLASS region channel promo / param=reference` deklaruje kategoriální prediktory s referenčním kódováním.
- `selection=stepwise(select=sbc choose=sbc)` řídí hledání.
- `details=summary` vypisuje krok za krokem souhrn výběru; `stb` přidává standardizované koeficienty, aby byly efekty na různých škálách porovnatelné.
- `output out=demand_scored p=predicted r=residual` uloží predikované hodnoty a rezidua pro další skórování.

**Souhrn krokového výběru** čtěte jako záznam hledání: začíná od plného 12efektového modelu a postupně efekty *odebírá* — postupně vyřadí `noise1`, `noise2`, `temp_f`, kontrast `Region=Midwest` a `noise3`, protože každé odebrání snižuje SBC. Co zůstane v tabulce **Odhady parametrů**, je zvolený model.

In [3]:
PROCEDURA glmselect data=demand seed=20260531;
    TŘÍDA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=stepwise(select=sbc choose=sbc)
          details=summary stb;
    VÝSTUP out=demand_scored p=predicted r=residual;
    NÁZEV 'Krokový výběr faktorů poptávky (SBC)';
SPUSTIT;

                                         Týdenní poptávka a kandidátní faktory                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Prodané kusy


Number of Observations Used: 100

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                Stepwise Selection Summary                                                 

    Step    Action          Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)
--------  --------  --------------  -----------------  --------  --------  --------  --------  --------  --------  --------
       1   Removed          NOISE1                 12    0.9507    0.9439  707.0094  711.2420  713.1843  740.8766   12.0136
       2   Rem


NOTE: PROC GLMSELECT data=demand

NOTE: The data set WORK.DEMAND_SCORED has 100 observations and 15 variables.
NOTE: PROC GLMSELECT statement used.


## 4. LASSO s křížovou validací

LASSO smršťuje koeficienty k nule a provádí tak výběr a regularizaci současně. Místo zastavení na pevném kritériu necháme **5násobnou křížovou validaci**, aby vybrala bod na cestě LASSO s nejlepší chybou mimo trénovací vzorek.

- `selection=lasso(choose=cv stop=none)` sleduje celou cestu LASSO a vybere CV-optimální krok.
- `cvmethod=random(5)` přiřadí pozorování do 5 náhodných foldů.

**Souhrn výběru LASSO** ukazuje pořadí, v jakém efekty vstupují, jak se penalizace uvolňuje: nejdříve `price`, poté `adspend`, `competitor`, region Northeast, `email`, `promo` a `holiday` — všech sedm skutečných signálů vstoupí dříve než jakákoli rušivá proměnná. Křížová validace poté zvolí krok s nejnižším PRESS mimo trénovací vzorek a výsledná tabulka **Odhady parametrů** ponechá právě skutečné faktory (plus malý člen `Region=Midwest`), zatímco vyloučí `temp_f`, `noise1`, `noise2` a `noise3`, které vstupují až na samém konci cesty.

In [4]:
PROCEDURA glmselect data=demand seed=20260531;
    TŘÍDA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=lasso(choose=cv stop=none)
          cvmethod=random(5);
    NÁZEV 'LASSO s 5násobnou křížovou validací';
SPUSTIT;

                                         Týdenní poptávka a kandidátní faktory                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Prodané kusy


Number of Observations Used: 100

  Cross Validation Information   

                   Item     Value
-----------------------  --------
Cross Validation Method    Random
        Number of Folds         5

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                         LASSO Selection Summary                                                          

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  -----------------  ---


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 5. Dopředný výběr ověřený na zadržovaném vzorku

Třetí, doplňková strategie: sestavit model pomocí **dopředného výběru** (efekty pouze vstupují, nikdy neopouštějí model), ale zastavit se v bodě, kde je nejlepší výkonnost na nezávislém zadržovaném vzorku — což přímo chrání proti přeučení.

- `partition fraction(validate=0.30)` náhodně vyhradí 30 % řádků pro ověření, takže zbývá 70 trénovacích a 30 ověřovacích pozorování.
- `selection=forward(select=aic choose=validate)` přidává efekty podle AIC, ale finální model volí podle chyby na ověřovacím vzorku.

Tabulka **Podíly rozdělení** potvrzuje rozdělení 70/30. Dopředný výběr poté přidává efekty, dokud se ověřovací chyba zlepšuje; osm efektů ve finální tabulce **Odhady parametrů** jsou přesně skutečné faktory, čtyři rušivé proměnné nejsou nikdy přijaty. Když tři metody založené na odlišných principech dospějí ke stejným faktorům, je mnohem pravděpodobnější, že jde o skutečný nález, než artefakt jediného pravidla výběru.

In [5]:
PROCEDURA glmselect data=demand seed=20260531;
    TŘÍDA region channel promo / PARAM=reference;
    MODEL units = region channel promo price competitor adspend
                  email temp_f holiday noise1 noise2 noise3
        / selection=forward(select=aic choose=validate);
    partition fraction(validate=0.30);
    NÁZEV 'Dopředný výběr ověřený na 30% zadržovaném vzorku';
SPUSTIT;

                                         Týdenní poptávka a kandidátní faktory                                          

The GLMSELECT Procedure


Dependent Variable: UNITS Prodané kusy


Number of Observations Used: 70               
Number of Observations Used for Validation: 30

Partition Fractions 

      Role  Fraction
----------  --------
  Training    0.7000
Validation    0.3000
   Testing    0.0000

          Class Level Information          

   Class    Levels                   Values
--------  --------  -----------------------
  region         3  Midwest Northeast South
 channel         2             Online Store
   promo         2                   No Yes

                                                        Forward Selection Summary                                                         

    Step    Action            Effect  Number Effects In  R-Square  Adj R-Sq       AIC      AICC       BIC       SBC      C(p)        PRESS
--------  --------  ----------------  -----


NOTE: PROC GLMSELECT data=demand

NOTE: PROC GLMSELECT statement used.


## 6. Interpretace výsledků

Všechny tři strategie výběru nacházejí **stejnou sadu skutečných faktorů poptávky** a vyřazují všechny rušivé proměnné. Při čtení přímo z tabulek **Odhady parametrů**:

- **Cena** je dominantním faktorem. Její standardizovaný koeficient v krokovém modelu je **-0.70**, zdaleka největší co do velikosti, a surový sklon se pohybuje mezi **-27.8** (krokový výběr a LASSO) a **-28.6** (dopředný výběr) jednotek na dolar — téměř přesně -28, se kterým byla data generována. **Cena konkurence** posouvá poptávku opačným směrem o zhruba **+14.4** u všech tří modelů, což je substituční chování, jaké kategorijní manažer očekává.
- **Výdaje na reklamu** (asi +0.062 kusu na dolar) a **objem e-mailů** (asi +1.2 kusu na zaslaný e-mail) obě zvyšují prodeje a kvantifikují mediální odezvu.
- **Akce** a **svátky** nesou největší efekty spojené s událostmi: kontrast `Promo=No` je přibližně **-51 až -57** vůči týdnu s akcí a nárůst o svátku je zhruba **+66 až +76** kusů. **Region Northeast** (asi +49 až +55) a **kanál Online** (asi +24 až +32) nesou kladné základní efekty.
- Zásadní je, že každá rušivá proměnná — `temp_f`, `noise1`, `noise2`, `noise3` — je krokovým i dopředným výběrem **vyřazena** a je vyloučena i z modelu LASSO zvoleného křížovou validací. Každá metoda odhalila strukturální signál a ignorovala šum.

Tyto tři metody nejsou identické do posledního detailu: krokový výběr (SBC) a dopředné hledání ověřené na zadržovaném vzorku se shodují na stejných osmi efektech, zatímco křížově validované LASSO navíc ponechává malý kontrast `Region=Midwest` (-8.3, směrodatná chyba 8.3) — rozdíl na úrovni šumu, nikoli věcný nesouhlas. Že seznam faktorů obstojí v krokovém SBC výběru, křížově validovaném LASSO i v dopředném výběru ověřeném na zadržovaném vzorku, je hlavní ponaučení: nález odolný vůči třem odlišným filozofiím výběru je mnohem důvěryhodnější než nález vyladěný na jediné kritérium. S `OUTPUT OUT=demand_scored`, které zachycuje predikované hodnoty a rezidua, se stejný postup přirozeně rozšiřuje na skórování plánovaného cenového a akčního kalendáře příštího čtvrtletí.